In [22]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from scipy.spatial import cKDTree

# ── 1. Load data ───────────────────────────────────────────────────────────────
print("Loading data...")
row_gdf = gpd.read_file(r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_ROW_Centroids.shp")
print(f"  ROW points loaded:       {len(row_gdf):,} records")
sub_gdf = gpd.read_file(r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Substations_InService_NotAvailable.shp")
print(f"  Substations loaded:      {len(sub_gdf):,} records")

# ── 2. Check and drop null geometries ─────────────────────────────────────────
print("\nChecking for null geometries...")
row_nulls = row_gdf.geometry.isna().sum()
sub_nulls = sub_gdf.geometry.isna().sum()
print(f"  Null geometries in ROW:          {row_nulls:,}")
print(f"  Null geometries in Substations:  {sub_nulls:,}")

row_gdf = row_gdf[row_gdf.geometry.notna()].reset_index(drop=True)
sub_gdf = sub_gdf[sub_gdf.geometry.notna()].reset_index(drop=True)
print(f"  ROW points after cleaning:       {len(row_gdf):,} records")
print(f"  Substations after cleaning:      {len(sub_gdf):,} records")

# ── 3. Reproject ──────────────────────────────────────────────────────────────
print("\nReprojecting to EPSG:...")
CRS = "EPSG:6634"
row_gdf = row_gdf.to_crs(CRS)
sub_gdf = sub_gdf.to_crs(CRS)
print("  Reprojection complete")

# ── 4. Build KD-Tree ──────────────────────────────────────────────────────────
print("\nBuilding KD-Tree from substation coordinates...")
sub_coords = np.array([(geom.x, geom.y) for geom in sub_gdf.geometry])
row_coords = np.array([(geom.x, geom.y) for geom in row_gdf.geometry])
tree = cKDTree(sub_coords)
print(f"  KD-Tree built from {len(sub_coords):,} substation points")

# ── 5. Find nearest substation ────────────────────────────────────────────────
print(f"\nFinding nearest substation for {len(row_coords):,} ROW points...")
distances, indices = tree.query(row_coords, k=1)
print(f"  Nearest neighbor search complete")
print(f"  Min distance:  {distances.min():,.2f} m")
print(f"  Max distance:  {distances.max():,.2f} m")
print(f"  Mean distance: {distances.mean():,.2f} m")

# ── 6. Build connecting lines ─────────────────────────────────────────────────
print("\nBuilding connecting lines...")
lines = [
    LineString([row_gdf.geometry.iloc[i], sub_gdf.geometry.iloc[indices[i]]])
    for i in range(len(row_gdf))
]
print(f"  {len(lines):,} lines created")

# ── 7. Export CSV ─────────────────────────────────────────────────────────────
print("\nExporting CSV...")
output_df = pd.DataFrame({
    "Facility_I":                   row_gdf["Facility_I"],
    "Substation_Distance_Meters":   np.round(distances, 2)
})
csv_path = r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_ROW_substation_distances.csv"
output_df.to_csv(csv_path, index=False)
print(f"  CSV exported: {csv_path}")

# ── 8. Export lines shapefile ─────────────────────────────────────────────────
print("\nExporting lines shapefile...")
lines_gdf = gpd.GeoDataFrame(output_df, geometry=lines, crs=CRS)
shp_path = r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_Substation_Lines.shp"
lines_gdf.to_file(shp_path)
print(f"  Shapefile exported: {shp_path}")

# ── 9. Summary ────────────────────────────────────────────────────────────────
print("\n── Summary ───────────────────────────────────────────")
print(f"  Total ROW points processed:  {len(output_df):,}")
print(f"  Null ROW geometries dropped: {row_nulls:,}")
print(f"  Null SUB geometries dropped: {sub_nulls:,}")
print(f"  Min distance:                {distances.min():,.2f} m")
print(f"  Max distance:                {distances.max():,.2f} m")
print(f"  Mean distance:               {distances.mean():,.2f} m")
print("──────────────────────────────────────────────────────")
print(output_df.head())

Loading data...
  ROW points loaded:       30 records
  Substations loaded:      75,328 records

Checking for null geometries...
  Null geometries in ROW:          0
  Null geometries in Substations:  1
  ROW points after cleaning:       30 records
  Substations after cleaning:      75,327 records

Reprojecting to EPSG:...
  Reprojection complete

Building KD-Tree from substation coordinates...
  KD-Tree built from 75,327 substation points

Finding nearest substation for 30 ROW points...
  Nearest neighbor search complete
  Min distance:  825.27 m
  Max distance:  8,115.14 m
  Mean distance: 3,355.21 m

Building connecting lines...
  30 lines created

Exporting CSV...
  CSV exported: C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_ROW_substation_distances.csv

Exporting lines shapefile...
  Shapefile exported: C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_Substation_Lines.shp

── Summary ───────────────────────────────────────────
  Tot

C:\Users\KyleSteen.AzureAD\AppData\Local\Temp\ipykernel_20052\2864752068.py:70: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  lines_gdf.to_file(shp_path)
C:\Users\KyleSteen.AzureAD\miniconda3\envs\myenv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Substation_Distance_Meters' to 'Substation'
  ogr_write(


In [23]:
import geopandas as gpd
import pandas as pd

# ── INPUT PATHS ────────────────────────────────────────────────────────────────
fc_path = r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_Level_2_Analysis_AEP_GlintGlare_LCOE_Shape_Size.gpkg"
csv_path = r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_ROW_substation_distances.csv"

output_path = r"C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_substations_output.gpkg"

# ── 1. LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading data...")

fc_gdf = gpd.read_file(fc_path)
csv_df = pd.read_csv(csv_path)

print(f"  Feature class rows: {len(fc_gdf):,}")
print(f"  CSV rows:           {len(csv_df):,}")

# ── 2. CLEAN KEYS ──────────────────────────────────────────────────────────────
print("\nCleaning keys...")

fc_gdf["Facility_ID"] = fc_gdf["Facility_ID"].astype(str).str.strip()
csv_df["Facility_ID"] = csv_df["Facility_ID"].astype(str).str.strip()

# remove trailing .0 if present
fc_gdf["Facility_ID"] = fc_gdf["Facility_ID"].str.replace(r"\.0$", "", regex=True)
csv_df["Facility_ID"] = csv_df["Facility_ID"].str.replace(r"\.0$", "", regex=True)

# ensure CSV uniqueness (1 value per Facility_ID)
csv_df = csv_df.drop_duplicates(subset="Facility_ID")

print(f"  Unique CSV keys: {len(csv_df):,}")

# ── 3. BUILD LOOKUP ────────────────────────────────────────────────────────────
print("\nBuilding lookup table...")

lookup = dict(zip(
    csv_df["Facility_ID"],
    csv_df["Substation_Distance_Meters"]
))

# ── 4. APPLY JOIN ──────────────────────────────────────────────────────────────
print("Applying distance values...")

fc_gdf["Substation_Distance_Meters"] = fc_gdf["Facility_ID"].map(lookup)

# ── 5. CHECK RESULTS ───────────────────────────────────────────────────────────
missing = fc_gdf["Substation_Distance_Meters"].isna().sum()

print("\n── Summary ───────────────────────────────────────────")
print(f"  Total rows:     {len(fc_gdf):,}")
print(f"  Matched rows:   {len(fc_gdf) - missing:,}")
print(f"  Unmatched rows: {missing:,}")

if fc_gdf["Substation_Distance_Meters"].notna().any():
    print(f"  Min distance:   {fc_gdf['Substation_Distance_Meters'].min():,.2f}")
    print(f"  Max distance:   {fc_gdf['Substation_Distance_Meters'].max():,.2f}")

# ── 6. SAVE OUTPUT ────────────────────────────────────────────────────────────
print("\nSaving output...")

fc_gdf.to_file(output_path, layer="updated_fc", driver="GPKG")

print(f"Done → {output_path}")

Loading data...
  Feature class rows: 110
  CSV rows:           30

Cleaning keys...
  Unique CSV keys: 30

Building lookup table...
Applying distance values...

── Summary ───────────────────────────────────────────
  Total rows:     110
  Matched rows:   110
  Unmatched rows: 0
  Min distance:   825.27
  Max distance:   8,115.14

Saving output...
Done → C:\Users\KyleSteen.AzureAD\Documents\Substation_Workspace\Hawaii\Hawaii_substations_output.gpkg
